# Training EfficientNetB0 - Klasifikasi Penyakit Kelapa Sawit

**Dataset:** Kaggle Oil Palm Disease / Roboflow

**Kelas:** Ganoderma, Blight, Crown Disease, Pestalotiopsis, Basal Stem Rot, Spear Rot, Healthy

**Model:** EfficientNetB0 (transfer learning dari ImageNet)

In [ ]:
# Install dependencies (jalankan sekali)
# !pip install torch torchvision timm scikit-learn matplotlib seaborn kaggle

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR   = './data'          # struktur: data/train/ClassName/, data/val/ClassName/
MODEL_PATH = './models/efficientnet_sawit.pt'
BATCH_SIZE = 32
EPOCHS     = 20
LR         = 1e-4
IMG_SIZE   = 224

CLASSES = ['Basal Stem Rot', 'Blight', 'Crown Disease',
           'Ganoderma', 'Healthy', 'Pestalotiopsis', 'Spear Rot']

print(f'Device: {DEVICE}')

## 1. Download Dataset

Pilih salah satu sumber:

In [ ]:
# Opsi A: Kaggle
# !kaggle datasets download -d your-dataset-name -p ./data --unzip

# Opsi B: Roboflow (ganti dengan URL project kamu)
# !pip install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_KEY')
# project = rf.workspace().project('oil-palm-disease')
# dataset = project.version(1).download('folder')

# Struktur folder yang dibutuhkan:
# data/
#   train/
#     Ganoderma/  img1.jpg img2.jpg ...
#     Blight/     ...
#     Healthy/    ...
#     ...
#   val/
#     Ganoderma/  ...
#     ...
print('Pastikan struktur folder data/ sudah benar sebelum lanjut')

## 2. Data Preprocessing & Augmentation

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_tf)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'),   transform=val_tf)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f'Kelas: {train_ds.classes}')

## 3. Build Model (EfficientNetB0 Transfer Learning)

In [ ]:
def build_model(num_classes: int) -> nn.Module:
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    # Freeze semua layer kecuali classifier
    for param in model.features.parameters():
        param.requires_grad = False
    # Ganti classifier head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model.to(DEVICE)

model = build_model(len(train_ds.classes))
print(f'Model siap. Output classes: {len(train_ds.classes)}')

## 4. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # --- Validation ---
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            val_loss += criterion(out, labels).item()
            correct  += (out.argmax(1) == labels).sum().item()
            total    += labels.size(0)

    acc = correct / total
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss / len(train_dl))
    history['val_loss'].append(val_loss / len(val_dl))
    history['val_acc'].append(acc)

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | '
          f'Train Loss: {train_loss/len(train_dl):.4f} | '
          f'Val Loss: {val_loss/len(val_dl):.4f} | '
          f'Val Acc: {acc:.4f}')

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  ✓ Model tersimpan (acc={best_acc:.4f})')

print(f'\nBest Val Accuracy: {best_acc:.4f}')

## 5. Evaluasi & Visualisasi

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history['val_acc'])
ax2.set_title('Val Accuracy')
plt.tight_layout()
plt.savefig('./models/training_history.png')
plt.show()

In [ ]:
# Confusion matrix
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_dl:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=train_ds.classes))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=train_ds.classes, yticklabels=train_ds.classes)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('./models/confusion_matrix.png')
plt.show()

In [ ]:
# Simpan class mapping untuk dipakai di backend
class_map = {str(v): k for k, v in train_ds.class_to_idx.items()}
with open('./models/class_map.json', 'w') as f:
    json.dump(class_map, f, indent=2)
print('class_map.json tersimpan:', class_map)